<div style="display: flex; justify-content: space-between; align-items: center;">
    <div style="text-align: left; flex: 4;">
        <strong>Author:</strong> Amirhossein Heydari — 
        📧 <a href="mailto:amirhosseinheydari78@gmail.com">amirhosseinheydari78@gmail.com</a> — 
        🐙 <a href="https://github.com/mr-pylin/data-visualization-workshop" target="_blank" rel="noopener">github.com/mr-pylin</a>
    </div>
    <div style="display: flex; justify-content: flex-end; flex: 1; gap: 8px; align-items: center; padding: 0;">
        <a href="https://matplotlib.org/" target="_blank" rel="noopener noreferrer">
            <img src="../../assets/images/libraries/matplotlib/logo/Matplotlib_icon.svg"
                 alt="Matplotlib Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://seaborn.pydata.org/" target="_blank" rel="noopener noreferrer">
            <img src="../../assets/images/libraries/seaborn/logo/logo-mark-lightbg.svg"
                 alt="Seaborn Logo"
                 style="max-height: 48px; width: auto;">
        </a>
        <a href="https://plotly.com/python/" target="_blank" rel="noopener noreferrer">
            <img src="../../assets/images/libraries/plotly/logo/Plotly-Logo-White copy.svg"
                 alt="Plotly Logo"
                 style="max-height: 48px; width: auto; background-color: #1f1f1f; border-radius: 8px;">
        </a>
    </div>
</div>
<hr>


**Table of contents**<a id='toc0_'></a>    
- [Dependencies](#toc1_)    
- [Animation Interface](#toc2_)    
  - [FuncAnimation](#toc2_1_)    
    - [Basic Usage](#toc2_1_1_)    
    - [Using OOP-Style](#toc2_1_2_)    
    - [Frame Update Functions](#toc2_1_3_)    
    - [Saving Animations (GIF, MP4, HTML)](#toc2_1_4_)    
  - [Interactive Widgets](#toc2_2_)    
    - [Integrating with Jupyter Widgets](#toc2_2_1_)    
    - [Sliders, Buttons, and Dropdowns for Real-Time Parameter Control](#toc2_2_2_)    
  - [Event Handling and Callbacks](#toc2_3_)    
    - [Mouse and Keyboard Events](#toc2_3_1_)    
    - [Responding to Clicks, Hover, and Key Presses](#toc2_3_2_)    
    - [Interactive Plot Adjustments via Callbacks](#toc2_3_3_)    
  - [Cursor](#toc2_4_)    
    - [Cursor, SnaptoCursor, and MultiCursor](#toc2_4_1_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# <a id='toc1_'></a>[Dependencies](#toc0_)

In [ ]:
import shutil
from collections.abc import Callable
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML, display
from matplotlib import ExecutableNotFoundError
from matplotlib.animation import FFMpegWriter, FuncAnimation, ImageMagickWriter, PillowWriter
from matplotlib.widgets import Cursor, MultiCursor
from numpy.typing import NDArray

In [ ]:
# disable automatic figure display (plt.show() required)
# this ensures consistency with .py scripts
plt.ioff()

In [ ]:
output_dir = Path("../../output/matplotlib/06")
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
display_backend = FuncAnimation.to_html5_video if shutil.which("ffmpeg") else FuncAnimation.to_jshtml

# log
print(display_backend)

# <a id='toc2_'></a>[Animation Interface](#toc0_)


## <a id='toc2_1_'></a>[FuncAnimation](#toc0_)

- `FuncAnimation` from `matplotlib.animation` enables **creating animations by repeatedly updating a plot**.
- It is the primary tool for animating lines, scatter points, and other artists.
- You can visualize the animation in several ways depending on your environment and available backends.


🎞️ **Displaying the Animation**

- **In Jupyter / IPython environments**
  - Use one of the built-in HTML conversion methods to embed the animation inline

    <table>
        <thead>
        <tr>
            <th>Method</th>
            <th>Output format</th>
            <th>Dependencies</th>
            <th>File size</th>
            <th>Notes</th>
        </tr>
        </thead>
        <tbody>
        <tr>
            <td>to_jshtml()</td>
            <td>HTML + JavaScript + PNG frames</td>
            <td>None</td>
            <td>Larger</td>
            <td>Pure Python, slower to load for long animations</td>
        </tr>
        <tr>
            <td>to_html5_video()</td>
            <td>HTML5 &lt;video&gt; (MP4)</td>
            <td>FFmpeg</td>
            <td>Smaller</td>
            <td>Compact and efficient for export</td>
        </tr>
        </tbody>
    </table>

- **In Standard Python (Non-Jupyter)**
  - You can play the animation using an interactive GUI backend

    <table>
        <thead>
        <tr>
            <th>Backend</th>
            <th>Platform</th>
            <th>Available by default</th>
            <th>Notes</th>
        </tr>
        </thead>
        <tbody>
        <tr>
            <td>TkAgg</td>
            <td>Cross-platform</td>
            <td>✅ Usually yes</td>
            <td>Uses built-in Tkinter</td>
        </tr>
        <tr>
            <td>MacOSX</td>
            <td>macOS</td>
            <td>✅</td>
            <td>Native Cocoa backend</td>
        </tr>
        <tr>
            <td>Qt5Agg / Qt6Agg</td>
            <td>Cross-platform</td>
            <td>❌ Requires PyQt5/PyQt6</td>
            <td>Modern UI backend</td>
        </tr>
        </tbody>
    </table>


📝 Docs:
- `matplotlib.animation.FuncAnimation`: [matplotlib.org/stable/api/animation_api.html#matplotlib.animation.FuncAnimation](https://matplotlib.org/stable/api/animation_api.html#matplotlib.animation.FuncAnimation)
- Decay: [matplotlib.org/stable/gallery/animation/animate_decay.html](https://matplotlib.org/stable/gallery/animation/animate_decay.html)


### <a id='toc2_1_1_'></a>[Basic Usage](#toc0_)

- Requires a **figure**, an **update function**, and a **frame sequence**
- The update function modifies plot elements for each frame
- Can animate **line plots, scatter plots, bar charts, or images**
- `frames` specifies number of frames or data sequence
- `interval` sets **delay between frames in milliseconds**
- `repeat` controls whether animation loops continuously


In [ ]:
x = np.linspace(-2, 2, 400)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
(line,) = ax.plot([], [], lw=2)
ax.set(xlim=[np.min(x), np.max(x)], ylim=[-2, 2])

In [ ]:
def init() -> tuple:
    line.set_data([], [])
    return (line,)


def update(frame: int) -> tuple:
    y = np.sin(x + frame / 10.0)
    line.set_data(x, y)
    ax.set_title(f"frame = {frame}")
    return (line,)


anim = FuncAnimation(
    fig,
    update,
    frames=np.arange(0, 200),
    init_func=init,
    interval=30,
    blit=True,
    repeat=True,
)
HTML(display_backend(anim))

### <a id='toc2_1_2_'></a>[Using OOP-Style](#toc0_)


In [ ]:
class FuncAnimator:

    def __init__(
        self,
        func: Callable[[NDArray, int], NDArray],
        x: np.ndarray,
        ylim: tuple,
        frames: int = 200,
        interval: int = 30,
        figsize: tuple = (5, 4),
        title_template: str = "Frame = {frame}",
    ):

        self.func = func
        self.x = x
        self.frames = frames
        self.interval = interval
        self.title_template = "" if not title_template else title_template

        # create figure and line
        self.fig, self.ax = plt.subplots(figsize=figsize)
        (self.line,) = self.ax.plot([], [], lw=2)
        self.ax.set(xlim=(np.min(x), np.max(x)), ylim=ylim)

        # FuncAnimation object
        self.anim = FuncAnimation(
            self.fig,
            self._update,
            frames=np.arange(frames) if isinstance(frames, int) else frames,
            init_func=self._init,
            interval=self.interval,
            blit=True,
            repeat=True,
        )

    def _init(self) -> tuple:
        self.line.set_data([], [])
        return (self.line,)

    def _update(self, frame: int) -> tuple:
        y = self.func(self.x, frame)
        self.line.set_data(self.x, y)
        self.ax.set_title(self.title_template.format(frame=frame))
        return (self.line,)

    def show(self, backend: Callable) -> HTML:
        return HTML(backend(self.anim))

In [ ]:
x = np.linspace(-2, 2, 400)


def sine_func(x: NDArray, frame: int) -> NDArray:
    return np.sin(x + frame / 10.0)


animator = FuncAnimator(
    sine_func,
    x,
    ylim=(-2, 2),
    frames=200,
    interval=30,
    title_template="Frame {frame}",
)
animator.show(display_backend)

### <a id='toc2_1_3_'></a>[Frame Update Functions](#toc0_)

- The update function takes a **frame index** (or data) as input
- Returns an iterable of modified artists for efficient rendering
- Optional `init_func` can set the initial state of the plot


In [ ]:
fig2, ax2 = plt.subplots(figsize=(5, 4), layout="compressed")
ax2.set(xlim=(-1, 1), ylim=(-1, 1), xticks=[], yticks=[])
(point,) = ax2.plot([], [], "o")
(trail,) = ax2.plot([], [], "-", alpha=0.6)

In [ ]:
xs, ys = [], []

In [ ]:
def generator_frames():
    t = 0.0
    while True:
        t += 0.1
        yield t


def init2():
    xs.clear()
    ys.clear()
    return point, trail


def update2(t):
    x = 0.6 * np.cos(t)
    y = 0.9 * np.sin(2 * t)
    xs.append(x)
    ys.append(y)
    point.set_data([x], [y])
    trail.set_data(xs[-60:], ys[-60:])
    return point, trail


anim2 = FuncAnimation(
    fig2,
    update2,
    frames=generator_frames,
    init_func=init2,
    interval=30,
    blit=True,
    save_count=200,
    repeat=True,
)
HTML(display_backend(anim2))

### <a id='toc2_1_4_'></a>[Saving Animations (GIF, MP4, HTML)](#toc0_)

- Animations can be saved as **MP4**, **GIF**, or **HTML**.
- These animations can be exported using suitable writer backends such as `ffmpeg`, `imagemagick` or `pillow`.
- The default writer is `ffmpeg`.

📝 **Docs**:

- `matplotlib.animation.Animation.save`: [matplotlib.org/stable/api/_as_gen/matplotlib.animation.animation](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.animation#matplotlib.animation.Animation.save)
- `matplotlib.animation.MovieWriter`: [matplotlib.org/stable/api/_as_gen/matplotlib.animation.moviewriter](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.moviewriter#matplotlib.animation.MovieWriter)


In [ ]:
# using ffmpeg (default) (if available)
try:
    anim2.save(filename=output_dir / "anim2_ffmpeg.gif", writer=FFMpegWriter(fps=30))
except ExecutableNotFoundError as e:
    print(e)

In [ ]:
# using Pillow (if available)
try:
    anim2.save(filename=output_dir / "anim2_pillow.gif", writer=PillowWriter(fps=30))
except ExecutableNotFoundError as e:
    print(e)

In [ ]:
# using imagemagick (if available)
try:
    anim2.save(filename=output_dir / "anim2_imagemagick.gif", writer=ImageMagickWriter(fps=30))
except ExecutableNotFoundError as e:
    print(e)

## <a id='toc2_2_'></a>[Interactive Widgets](#toc0_)

- Interactive widgets allow **real-time control of plot parameters** within Jupyter notebooks, enabling dynamic exploration of data.

📝 Docs:

- `ipywidgets`: [ipywidgets.readthedocs.io/en/stable/](https://ipywidgets.readthedocs.io/en/stable/)
- Backends: [matplotlib.org/stable/users/explain/backends.html#notebook-backends](https://matplotlib.org/stable/users/explain/backends.html#notebook-backends)
- Slider: [matplotlib.org/stable/gallery/widgets/slider_demo.html](https://matplotlib.org/stable/gallery/widgets/slider_demo.html)
- Simple Widget Introduction: [ipywidgets.readthedocs.io/en/stable/examples/Widget%20Basics.html#Callback-functions](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20Basics.html#Callback-functions)


In [ ]:
%matplotlib widget
plt.ioff()

### <a id='toc2_2_1_'></a>[Integrating with Jupyter Widgets](#toc0_)

- Use `ipywidgets` sliders, buttons, checkboxes, and dropdowns to adjust plot values
- Combine with Matplotlib to **update figures interactively**
- Requires `%matplotlib widget` backend for live interactivity in notebooks


In [ ]:
# create the figure and axis
fig_w, ax_w = plt.subplots()

# initialize the plot
x = np.linspace(0, 2 * np.pi, 400)
(line_w,) = ax_w.plot(x, np.sin(x))
ax_w.set_ylim(-3, 3)

In [ ]:
# create sliders
amp_slider = widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description="amp")
freq_slider = widgets.FloatSlider(value=1.0, min=0.0, max=5.0, step=0.1, description="freq")

In [ ]:
def update_plot(change=None):
    a = amp_slider.value
    f = freq_slider.value
    line_w.set_ydata(a * np.sin(f * x))
    ax_w.set_title(f"amp={a:.2f}, freq={f:.2f}")
    fig_w.canvas.draw_idle()  # update the canvas instead of displaying the figure


amp_slider.observe(update_plot, names="value")
freq_slider.observe(update_plot, names="value")

In [ ]:
controls = widgets.HBox([amp_slider, freq_slider])
display(controls)
plt.show()

### <a id='toc2_2_2_'></a>[Sliders, Buttons, and Dropdowns for Real-Time Parameter Control](#toc0_)

- Sliders: adjust numeric parameters dynamically
- Buttons: trigger events or reset values
- Dropdowns: select among categorical options
- Link widgets to a **callback function** that updates the plot


In [ ]:
# create the figure and axis
fig_w, ax_w = plt.subplots()

# initialize the plot
x = np.linspace(0, 2 * np.pi, 400)
(line_w,) = ax_w.plot(x, np.sin(x))
ax_w.set_ylim(-3, 3)

In [ ]:
button = widgets.Button(description="Randomize")
drop = widgets.Dropdown(options=["sin", "cos", "tan"], value="sin")

In [ ]:
def on_button_clicked(b):
    amp_slider.value = np.random.uniform(0.2, 2.5)
    freq_slider.value = np.random.uniform(0.2, 4.0)


def on_dropdown_change(change):
    func = change["new"]
    if func == "sin":
        line_w.set_ydata(amp_slider.value * np.sin(freq_slider.value * x))
    elif func == "cos":
        line_w.set_ydata(amp_slider.value * np.cos(freq_slider.value * x))
    elif func == "tan":
        line_w.set_ydata(amp_slider.value * np.tan(freq_slider.value * x))


button.on_click(on_button_clicked)
drop.observe(on_dropdown_change, names="value")

In [ ]:
display(widgets.HBox([button, drop]))
plt.show()

## <a id='toc2_3_'></a>[Event Handling and Callbacks](#toc0_)

- Matplotlib supports **interactive events**, allowing plots to respond to mouse and keyboard input in real time.

📝 Docs:
- `FigureCanvas.mpl_connect`: [matplotlib.org/stable/api/backend_bases_api.html#matplotlib.backend_bases.FigureCanvasBase.mpl_connect](https://matplotlib.org/stable/api/backend_bases_api.html#matplotlib.backend_bases.FigureCanvasBase.mpl_connect)


### <a id='toc2_3_1_'></a>[Mouse and Keyboard Events](#toc0_)

- Connect event types (e.g., `button_press_event`, `key_press_event`, `motion_notify_event`) to callback functions
- Callback functions receive an **event object** with information about the event (coordinates, key pressed, button, etc.)
- Enables interactive zooming, selection, or custom behaviors


In [ ]:
fig_e, ax_e = plt.subplots()

ax_e.set_xlim(0, 10)
ax_e.set_ylim(0, 10)
sc = ax_e.scatter([], [])
pts = []


def onclick(event):
    # ignore clicks outside axes
    if event.inaxes != ax_e:
        return
    pts.append((event.xdata, event.ydata))
    ax_e.scatter(event.xdata, event.ydata, c="red")


def onpress(event):
    print(f"Key pressed: {event.key}")
    if event.key == "c":
        pts.clear()
        ax_e.cla()
        ax_e.set_xlim(0, 10)
        ax_e.set_ylim(0, 10)


cid_click = fig_e.canvas.mpl_connect("button_press_event", onclick)
cid_key = fig_e.canvas.mpl_connect("key_press_event", onpress)

plt.ion()
plt.show()

### <a id='toc2_3_2_'></a>[Responding to Clicks, Hover, and Key Presses](#toc0_)

- Clicks: add or modify points, trigger highlights
- Hover: display data values, tooltips, or dynamic annotations
- Key presses: toggle visibility, change plot parameters, or switch views


In [ ]:
fig_h, ax_h = plt.subplots()

ax_h.set_xlim(0, 5)
ax_h.set_ylim(0, 5)
annot = ax_h.annotate(
    "", xy=(0, 0), xytext=(15, 15), textcoords="offset points", bbox=dict(boxstyle="round", fc="gray")
)
annot.set_visible(False)


def motion(event):
    if event.inaxes == ax_h and event.xdata is not None:
        annot.xy = (event.xdata, event.ydata)
        annot.set_text(f"x={event.xdata:.2f}\ny={event.ydata:.2f}")
        annot.set_visible(True)
    else:
        annot.set_visible(False)


fig_h.canvas.mpl_connect("motion_notify_event", motion)
plt.show()

### <a id='toc2_3_3_'></a>[Interactive Plot Adjustments via Callbacks](#toc0_)

- Update plot elements within callbacks and refresh with `canvas.draw_idle()` for efficiency
- Combine with widgets for richer interactivity
- Can manage multiple event types simultaneously for complex interfaces


In [ ]:
fig_d, ax_d = plt.subplots()

ax_d.set_xlim(0, 10)
ax_d.set_ylim(0, 10)
(param_point,) = ax_d.plot(5, 5, "o", markersize=10, picker=5)
text_param = ax_d.text(0.02, 0.95, "param=5", transform=ax_d.transAxes)


dragging = {"active": False}


def on_pick(event):
    dragging["active"] = True


def on_release(event):
    dragging["active"] = False


def on_motion(event):
    if not dragging["active"] or event.inaxes != ax_d:
        return
    param_point.set_data(event.xdata, event.ydata)
    text_param.set_text(f"param={event.xdata:.2f}")


fig_d.canvas.mpl_connect("pick_event", on_pick)
fig_d.canvas.mpl_connect("button_release_event", on_release)
fig_d.canvas.mpl_connect("motion_notify_event", on_motion)


plt.show()

## <a id='toc2_4_'></a>[Cursor](#toc0_)

- Cursors and annotation tools enhance **interactive plot exploration** by dynamically displaying coordinates and highlighting points of interest.

📝 Docs:
- `matplotlib.widgets.Cursor`: [matplotlib.org/stable/api/widgets_api.html#matplotlib.widgets.Cursor](https://matplotlib.org/stable/api/widgets_api.html#matplotlib.widgets.Cursor)
- `matplotlib.widgets.MultiCursor`: [matplotlib.org/stable/api/widgets_api.html#matplotlib.widgets.MultiCursor](https://matplotlib.org/stable/api/widgets_api.html#matplotlib.widgets.MultiCursor)
- Cross-hair cursor: [matplotlib.org/stable/gallery/event_handling/cursor_demo.html](https://matplotlib.org/stable/gallery/event_handling/cursor_demo.html)


### <a id='toc2_4_1_'></a>[Cursor, SnaptoCursor, and MultiCursor](#toc0_)

- `Cursor`: shows crosshairs that follow the mouse over the plot
- `SnaptoCursor`: snaps the cursor to the nearest data point for precise reading
- `MultiCursor`: synchronizes cursors across multiple subplots for comparison


In [ ]:
fig_c, (axc1, axc2) = plt.subplots(2, 1, figsize=(6, 6), sharex=True)


t = np.linspace(0, 10, 500)
axc1.plot(t, np.sin(t))
axc2.plot(t, np.cos(t))


cursor = Cursor(axc1, useblit=True, horizOn=True, vertOn=True, linewidth=1)
# multicursor = MultiCursor(fig_c.canvas, (axc1, axc2), horizOn=True, vertOn=True)


plt.show()